# 05_feature_concat_model
KLUE-RoBERTa [CLS] 벡터 + 숫자 feature concat 분류

```
clean_text → RoBERTa → [CLS] (768d)
                              ↓
숫자 feature 5개 ──→ concat (773d) → Linear → Good/Poor
```

early stopping 추가

## 셀 1 — Drive 마운트 & 라이브러리 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch scikit-learn pandas numpy -q
print("설치 완료!")

## 셀 2 — 데이터 로드

In [ ]:
import pandas as pd
import numpy as np

base_path = '/content/drive/MyDrive/presentation_data/'

train_df = pd.read_csv(base_path + 'train.csv')
val_df   = pd.read_csv(base_path + 'val.csv')
test_df  = pd.read_csv(base_path + 'test.csv')

print(f"Train: {len(train_df)}개 | Val: {len(val_df)}개 | Test: {len(test_df)}개")
print(f"컬럼: {list(train_df.columns)}")
print(f"\nTrain 레이블 분포:")
print(train_df['label'].value_counts())

## 셀 3 — 숫자 feature 스케일링

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = ['total_words', 'filler_count', 'filler_ratio', 'vocab_diversity', 'avg_word_len']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[feature_cols])
X_val_scaled   = scaler.transform(val_df[feature_cols])
X_test_scaled  = scaler.transform(test_df[feature_cols])

print(f"숫자 feature 스케일링 완료! shape: {X_train_scaled.shape}")

## 셀 4 — 토크나이저 로드

In [ ]:
import torch
from transformers import AutoTokenizer

model_name = 'klue/roberta-small'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"토크나이저 로드 완료! | 디바이스: {device}")

## 셀 5 — Dataset (텍스트 + 숫자 feature 함께 전달)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ConcatDataset(Dataset):
    def __init__(self, df, num_features, tokenizer, max_len=128):
        self.texts        = df['clean_text'].fillna('').tolist()
        self.labels       = df['label'].tolist()
        self.num_features = num_features          # 스케일링된 숫자 feature (numpy array)
        self.tokenizer    = tokenizer
        self.max_len      = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'num_features':   torch.tensor(self.num_features[idx], dtype=torch.float),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ConcatDataset(train_df, X_train_scaled, tokenizer)
val_dataset   = ConcatDataset(val_df,   X_val_scaled,   tokenizer)
test_dataset  = ConcatDataset(test_df,  X_test_scaled,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print(f"Train 배치 수: {len(train_loader)} | Val 배치 수: {len(val_loader)}")
print("데이터셋 준비 완료!")

## 셀 6 — 모델 정의 (RoBERTa + 숫자 feature concat)

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class RoBERTaWithFeatures(nn.Module):
    def __init__(self, model_name, num_features=5, num_labels=2, dropout=0.1):
        super().__init__()
        self.roberta    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.roberta.config.hidden_size  # 768 (roberta-small: 768)

        self.dropout    = nn.Dropout(dropout)
        # [CLS](768) + 숫자feature(5) = 773 → 2
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + num_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, num_features, labels=None):
        outputs    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # [CLS] 토큰
        cls_output = self.dropout(cls_output)

        # concat: [CLS](768) + 숫자feature(5)
        combined   = torch.cat([cls_output, num_features], dim=1)
        logits     = self.classifier(combined)

        loss = None
        if labels is not None:
            weight = torch.tensor([2.5, 1.0]).to(labels.device) #변경
            loss = nn.CrossEntropyLoss(weight=weight)(logits, labels) #변경

        return loss, logits

model = RoBERTaWithFeatures(model_name).to(device)
print("모델 정의 완료!")
print(f"구조: RoBERTa [CLS](768) + 숫자feature(5) → Linear(256) → Linear(2)")

## 셀 7 — 옵티마이저 & 스케줄러

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

epochs      = 5
optimizer   = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * epochs
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f"총 학습 스텝: {total_steps}")

## 셀 8 — 학습 & 평가 함수

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        num_features   = batch['num_features'].to(device)
        label          = batch['label'].to(device)

        optimizer.zero_grad()
        loss, logits = model(input_ids, attention_mask, num_features, label)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds  += logits.argmax(dim=1).cpu().tolist()
        labels += label.cpu().tolist()

    return total_loss / len(loader), accuracy_score(labels, preds), f1_score(labels, preds)


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, preds, labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            num_features   = batch['num_features'].to(device)
            label          = batch['label'].to(device)

            loss, logits = model(input_ids, attention_mask, num_features, label)
            total_loss  += loss.item()
            preds  += logits.argmax(dim=1).cpu().tolist()
            labels += label.cpu().tolist()

    return total_loss / len(loader), accuracy_score(labels, preds), f1_score(labels, preds)

print("함수 정의 완료!")

## 셀 9 — 학습 루프

In [ ]:
save_path    = base_path + 'best_model_concat'
best_val_f1  = 0
best_val_loss = float('inf')   # ← 추가
patience      = 2               # ← 추가
patience_counter = 0            # ← 추가
history      = []

for epoch in range(epochs):
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss,   val_acc,   val_f1   = eval_epoch(model, val_loader, device)

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss, 'train_acc': train_acc, 'train_f1': train_f1,
        'val_loss':   val_loss,   'val_acc':   val_acc,   'val_f1':  val_f1
    })

    print(f"Epoch {epoch+1}/{epochs} "
          f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} "
          f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")

    # ↓ 기존 val_f1 저장 조건은 그대로 유지
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), base_path + 'best_model_concat.pt')
        print(f"  → 최고 모델 저장! Val F1: {val_f1:.4f}")

    # ↓ Early stopping: val_loss 기준으로 추가
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  ※ val_loss 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"  → Early stopping! Epoch {epoch+1}에서 종료")
            break

print(f"\n학습 완료! 최고 Val F1: {best_val_f1:.4f}")

## 셀 10 — 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', marker='o')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1', marker='o')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val F1',   marker='o')
axes[1].set_title('F1 Score')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + 'training_curve_concat.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장 완료!")

## 셀 11 — Test 셋 최종 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 최고 모델 로드
best_model = RoBERTaWithFeatures(model_name).to(device)
best_model.load_state_dict(torch.load(base_path + 'best_model_concat.pt'))

test_loss, test_acc, test_f1 = eval_epoch(best_model, test_loader, device)
print(f"=== Test 결과 ===")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"Test F1-score: {test_f1:.4f}")

# 상세 리포트
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        num_features   = batch['num_features'].to(device)
        _, logits      = best_model(input_ids, attention_mask, num_features)
        all_preds += logits.argmax(dim=1).cpu().tolist()
        all_labels += batch['label'].tolist()

print("\n" + classification_report(all_labels, all_preds, target_names=['Poor(0)', 'Good(1)']))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Poor(0)', 'Good(1)'],
            yticklabels=['Poor(0)', 'Good(1)'])
plt.title('Confusion Matrix (Test) — Concat Model')
plt.ylabel('실제')
plt.xlabel('예측')
plt.tight_layout()
plt.savefig(base_path + 'confusion_matrix_concat.png', dpi=150, bbox_inches='tight')
plt.show()
print("평가 완료!")